<a href="https://colab.research.google.com/github/pelineceburgun/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W02 — ML Task Framing
**Lane 2: Refresh / Content Opportunity Scoring**

This notebook maps my chosen lane onto the ML loop, per `framing-ml-problems/SKILL.md`.
Framing is written and reasoned by me; AI assistance was used to check the framing and
surface edge cases in the data dictionary, not to decide the frame itself.

## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring (Core lane, `flyrank-context`).

**Question this answers:** *"Which pages should be reviewed first — for refresh, expansion,
protection, pruning, or monitoring?"*

Per the task-type table in `framing-ml-problems/SKILL.md`, "which ones first?" maps to
**ranking / scoring**, not plain classification. Concretely, the deliverable is a **ranked
queue** of content items, produced by combining:

- a **scoring** component (a continuous priority score per content item), and
- an underlying **binary signal** (e.g., "is this content declining / underperforming given
  its exposure") that the score is built on, evaluated with ranking-appropriate metrics
  (precision@K) rather than generic accuracy.

So this is a **ranking/scoring task with a classification-shaped label underneath it** —
the label answers "is this a case worth flagging," and the score orders cases by how
strongly the evidence points that way, within a fixed review capacity (top-K).

## 2. Target or proxy

**Starter-data proxy (what I can build immediately):**
`is_declining_label = (trend_direction == "down")`

Per `flyrank-data/SKILL.md`, this is explicitly a **label trap** if handled carelessly:
`trend_direction` is derived from `trend_pct`, so **neither `trend_direction` nor `trend_pct`
can ever be used as a feature** — only as the label itself. I use it here as a *beginner
proxy target*, not the ideal capstone target, consistent with the lane guide's own framing
of it as a current-window derived bucket.

**Stronger target for later weeks (once I move to the warehouse):**
A future-observed outcome, e.g.features from prior 90 days (per-client window, respecting gsc_data_start/ga4_data_start)
-> decline or recovery over the next 30 days, measured directly from
fact_content_daily_performance (not from any pre-computed trend field)

This satisfies the "target must be observed, not defined" rule from `framing-ml-problems`:
a future outcome is measured, not asserted by an upstream rule.

## 3. Success metric

**Primary metric: Precision@K** (K = review capacity, e.g., top 50), because the actual
decision this supports is "which N pages does a reviewer look at first," not "classify
every page correctly." This matches how the starter pipeline itself was evaluated
(baseline rules Precision@50 = 0.240 vs. random forest Precision@50 = 0.740 in
`outputs/model_report.md`).

**Secondary metrics:**
- **Average precision** — whether the *whole* ranking is sensible, not just the top slice.
- **Recall at a fixed threshold** — because a missed real decline (false negative) has a
  real, if harder-to-see, cost: the client's page keeps losing visibility unnoticed.

**Explicitly not the primary metric:** plain accuracy or ROC-AUC alone — a reviewer never
reads the *whole* ranked list, so overall discrimination across all items is a weaker proxy
for the actual decision than precision at the top of the queue.

I am naming this now, before touching the warehouse data, per the "name the metric before
training" rule — so I can't quietly redefine "good" after seeing results.

## 4. The unit of analysis, as a real dataframe

One row = **one pseudonymized content item**, described by its trailing-90-day metrics
(starter dataset: `data/raw/content_refresh_anonymized.csv`, 30,000 rows × 44 columns,
32 clients).

Loading it below and applying the known gotchas from `flyrank-data/SKILL.md` before looking
at anything else:
- rate columns (`ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`) are
  already ×100 percentages — `ctr = 0.76` means 0.76%, not 76%;
- `avg_position == 0` means "no data," not "rank zero";
- `content_id` / `client_id` are pseudonyms — usable for grouping/joins/splits, never as
  model features.

In [ ]:
import pandas as pd

DATA_PATH = "https://raw.githubusercontent.com/pelineceburgun/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:
# Sanity checks flagged by the data dictionary / flyrank-data skill before doing anything else

# 1. avg_position == 0 means "no data", not rank zero
n_zero_position = (df["avg_position"] == 0).sum()
print(f"Rows with avg_position == 0 (no data, not rank 0): {n_zero_position}")

# 2. rate columns should look like small percentages (e.g. mostly well under 100),
#    but scroll_rate / ai_traffic_pct are explicitly allowed to exceed 100 -- not a bug
rate_cols = ["ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct"]
print(df[rate_cols].describe())

# 3. IDs are pseudonyms -- confirm they look like hash-style strings, not usable as features
print(df[["content_id", "client_id"]].head())
print("Unique clients:", df["client_id"].nunique())

Rows with avg_position == 0 (no data, not rank 0): 1205
                ctr  engagement_rate   scroll_rate  ai_traffic_pct  \
count  30000.000000     30000.000000  29875.000000    30000.000000   
mean       0.510733         2.534520     18.212921        0.768196   
std        3.279162         8.310096     29.472768        7.429454   
min        0.000000         0.000000      0.000000        0.000000   
25%        0.000000         0.000000      0.000000        0.000000   
50%        0.070000         0.000000      5.000000        0.000000   
75%        0.290000         1.350000     23.530000        0.000000   
max      100.000000       100.000000    300.000000      300.000000   

          trend_pct  
count  26612.000000  
mean      -4.785969  
std      473.861780  
min     -100.000000  
25%      -62.600000  
50%      -33.500000  
75%        0.000000  
max    44900.000000  
             content_id          client_id
0  content_304f48230142  client_f369cb89fc
1  content_a1fb4e703a9e  clie

In [ ]:
# One row = one content item. This IS the unit of analysis for the ranked queue.
unit_of_analysis_cols = [
    "content_id", "client_id", "content_type", "content_age_days",
    "impressions_90d", "clicks_90d", "sessions_90d", "avg_position",
    "ctr", "engagement_rate", "trend_direction",
]
df[unit_of_analysis_cols].head(10)

,content_id,client_id,content_type,content_age_days,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,187,3803,29,17,10.6,0.76,5.88,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,15320,7,9,20.3,0.05,0.00,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,12581,11,11,36.5,0.09,0.00,down
3,content_331d6c4de07b,client_19581e27de,keyword article,463,11751,58,78,6.2,0.49,1.28,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,19140,24,145,44.0,0.13,0.00,down
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,3970,1,5,8.5,0.03,0.00,down
6,content_9a34b442b552,client_8722616204,keyword article,90,20,0,1,7.0,0.00,0.00,down
7,content_a63219c6e95a,client_19581e27de,keyword article,445,1724,1,28,21.2,0.06,3.57,stable
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,32574,29,68,46.0,0.09,5.88,down
9,content_c27558df2b0c,client_19581e27de,keyword article,257,1240,2,3,4.9,0.16,0.00,down


## 5. Why ML beats a fixed rule here

Applying the four framing questions from `framing-ml-problems/SKILL.md`:

1. **What decision does this improve?** Which content items a reviewer with limited
   capacity looks at first this cycle — not "predict decline" in the abstract.
2. **Who acts, and what do they do?** A content strategist / SEO reviewer at FlyRank, who
   decides per flagged item: refresh, expand, protect, prune, or monitor.
3. **What does a wrong answer cost?** A false positive burns limited review time and
   possibly content-production budget on a page that didn't need it. A false negative lets
   a real decline run unnoticed. Mistaking **consolidation, seasonality, or noise** for real
   decline (lane guide, Section 7) is a specific, named failure mode here, not a generic one.
4. **Why does ML help, rather than a single if-statement?** Because "declining and worth
   reviewing" is not one clean signal — it's a *combination* of exposure level, position,
   CTR relative to position, engagement, freshness, and persistence over time, and the
   relative importance of these signals is not obvious by hand. This is exactly the "many
   signals, tangled, shifting over time" case the skill describes as where ML earns its
   place. Crucially, this claim is not just asserted here: the starter pipeline already
   shows a measured gap between a hand-written rule and a learned model on this exact
   problem shape (baseline Precision@50 = 0.240 vs. random forest Precision@50 = 0.740),
   which is evidence, not a guess, that the pattern is real but too messy for a fixed rule
   to fully capture.

**One-paragraph frame (per the skill's template):**

> For a FlyRank content strategist deciding which pages to review first this cycle, we will
> build a ranked review queue from trailing-90-day (later: prior-window) content and search
> signals, scoring/predicting whether a content item is declining or underperforming given
> its exposure, measured by precision@K (K = review capacity). A wrong call costs wasted
> reviewer time (false positive) or an unnoticed real decline (false negative), and
> mistaking consolidation/seasonality/noise for decline is a specific risk to guard against.
> A plain rule isn't enough because the relevant signals interact and are too numerous to
> hand-weight reliably, though a transparent rule still serves as the honest baseline to
> beat. We will claim only decision-support, ranked-review results — never that refreshing
> a flagged page is guaranteed to cause recovery.

## Self-check

- **Task type named?** Yes — ranking/scoring over a review-capacity-bounded queue, built on
  an underlying binary "worth flagging" signal.
- **Target/proxy named, and is it observed or defined?** Starter proxy
  (`trend_direction == "down"`) is *defined* by an upstream rule (a known limitation, stated
  explicitly, per the lane guide's own caveat) — not used as a feature source anywhere, only
  as the current-stage label. The Week 4+ target (future-window decline/recovery) will be
  *observed* directly from daily facts, satisfying the "target must be observed" rule.
- **Metric named before modeling, and computable today?** Yes — Precision@K, computable on
  the starter data's existing baseline/label pair without needing the warehouse.
- **Unit of analysis shown as a real dataframe?** Yes, above — one row = one pseudonymized
  content item over its trailing-90-day window.
- **Output tied to a real action?** Yes — a ranked queue with reason codes that a reviewer
  acts on (refresh/expand/protect/prune/monitor), not an autonomous action.
- **Where AI helped vs. where I framed it myself:** I used AI assistance to cross-check this
  framing against `framing-ml-problems/SKILL.md` and to catch data gotchas
  (`flyrank-data/SKILL.md`, e.g. the `trend_direction`/`trend_pct` label trap, the
  `avg_position == 0` meaning) before they became bugs. The decision, the action, the cost
  of a wrong call, and the choice of Precision@K over accuracy were my own calls, argued
  above, not generated for me.

**Open item to resolve before Week 4:** confirm the exact future-window target
(prior-90 → next-30, or a different split) once I've checked per-client history depth in
`dim_clients` against however many active-review-capacity clients are actually usable after
filtering by `access_profile` and `gsc_data_start`/`ga4_data_start`.

---

**Final checklist (per the skeleton):**

- [x] Every section above is filled — markdown reasoning **and** the code that backs it
- [ ] The notebook runs top to bottom with no errors (**Runtime → Run all** — to be confirmed
      in your environment once `DATA_PATH` points at the real file)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: *observed*, *measured*, *directional*, *decision-support*
- [ ] Committed to my repo under `work/notebooks/` — pending: commit after running locally